# **Leak Detection**: Regression with Gated Pipeline Harness
---

## **Value Proposition**

In order to catch data leakage before it reaches production, a gated ML lifecycle harness was applied to two grocery retail datasets. On the retail dataset, naive model evaluation scored R²=1.0000 — driven by a target-derived feature injected during preprocessing that correlated 1.0 with the target. The gated run on the same data returned R²=0.9294. On the Hill Country grocery dataset, the co-target detector found that `Weekly_Units_Sold × price` reconstructed weekly revenue to 0.989 correlation; after the harness dropped the co-target, the gated run scored R²=0.9078. In both cases, the naive score passed every standard quality check; only the harness gate sequence distinguished the inflated estimate from a deployable model.

### Business Opportunities

* **Opportunity A:** Preventing leaked models from reaching production, where they fail silently once the leaked feature is absent from live inference requests.
* **Solution A:** The retail gated run exposed a `deploy.smoke_test` failure — columns `te_Product_Type` and `avg_sales_for_product` existed at training time but not at inference time. Without gating, this model would have crashed on its first real request.

* **Opportunity B:** Detecting structural co-target leakage in datasets where a training column algebraically reconstructs the target.
* **Solution B:** The co-target detector flagged `Promo_Price_USD × Weekly_Units_Sold` with correlation 0.9886 on the Hill Country dataset. Dropping the co-target reduced R² from 0.9628 to 0.9078 — a 5.5-point gap that represents accounting arithmetic, not learned signal.

* **Opportunity C:** Standardizing pre-deployment validation across a team without manual audit.
* **Solution C** (Options): Embed the harness as a CI pre-merge gate on model training jobs. Run it as a post-training step in a feature store workflow. Use it as a lightweight audit on existing registered models before production promotion.

### Outcomes

* **Model Performance**
  * Retail gated: R²=0.9294, MAPE=4.31% on held-out test — Tier 1 (Exceptional)
  * Hill Country gated: R²=0.9078, MAPE=14.55% — Tier 2 (Strong, room to grow)
  * Naive runs scored R²=1.0000 and R²=0.9628; both cleared typical ship-it thresholds with no errors

* **Architecture**
  * Six candidates evaluated per run: Ridge, RandomForest, ExtraTrees, HistGBR, XGBoost, DecisionTree
  * RandomForest (max_depth=9) won the retail leaderboard; HistGBR (max_depth=12) won Hill Country after co-target removal

* **Economic Impact**
  * A retail model that scored R²=1.000 in evaluation and crashed on first production inference is an infrastructure reliability incident, not a model quality issue — rollback, audit, retrain, and re-validate typically costs 2-4 days of senior engineering time
  * A Hill Country demand forecast inflated by 5.5 R² points distorts safety-stock calculations and promotional planning before anyone detects the source

* **Strategy Recommendation**
  * **Enterprise:** Require gate-log artifacts alongside model artifacts in the model registry for all revenue-adjacent models; block promotion of any model with open gate failures
  * **SMB:** Run the harness as a one-step check before any model replaces a business-critical heuristic — pricing, ordering, or staffing decisions warrant the extra validation even in small teams

## **Problem Space**

### Overview

* Data leakage is the leading cause of overestimated model performance in tabular ML. Most patterns are invisible in aggregate metrics and only surface at deployment or through targeted gate checks.
* Two distinct leakage types are demonstrated: planted target-derived features (retail) and structural co-target leakage where training data contains a variable that algebraically reconstructs the target (Hill Country).
* A model with R²>0.90 on validation can be carrying leakage that guarantees production failure — the score provides no signal on its own.

### Data Description

* **8,763** retail observations and **8,880** Hill Country weekly grocery observations across 11 and 15 columns respectively
* Retail: product and store attributes; target `Product_Store_Sales_Total`; naive run injects 2 target-derived features (`te_Product_Type`, `avg_sales_for_product`) fit on the full dataset before the train/test split
* Hill Country: product, price, and store attributes; target `Weekly_Revenue_USD`; co-target `Weekly_Units_Sold` reconstructs target with correlation 0.9886 via `Promo_Price_USD × Weekly_Units_Sold`
* Data files: `retail_prediction.csv` and `hill_country_grocer_weekly_sales.csv`, loaded via `ensure_dataset("leak-detection__regression__gated-harness")`
* Both datasets have no zero-valued target rows, making MAPE valid on both

### Process

* Each dataset ran twice through the harness: first with gates in audit-only mode (naive), then with gates enforced (gated)
* In gated mode, the harness remediates before checking — identifiers and declared co-targets are dropped prior to any gate, so the reconstruction scan sees a clean feature set
* Model selection used a 6-candidate leaderboard ranked on a held-out validation set; the top candidate was hypertuned with automated grid backtrack on edge pinning

### Results

>| Run | Model | R² | MAPE | Gate Status |
>|---|---|---|---|---|
>| **Retail — Gated ✅** | RandomForest depth=9 | **0.9294** | 4.31% | All gates pass — lead deployment model |
>| Retail — Naive ⛔ | Ridge | 1.0000 | 0.00% | 5 gate failures (audit-only) |
>| **Hill Country — Gated ✅** | HistGBR depth=12 | **0.9078** | 14.55% | All gates pass — lead deployment model |
>| Hill Country — Naive ⛔ | RandomForest depth=15 | 0.9628 | 8.38% | 3 gate failures (audit-only) |

# **Code Execution**

### **Runtime Configuration**

> **Hardware Accelerator:** **CPU** and **Standard RAM**
>
> This notebook trains ensemble models on two datasets of ~9,000 rows each. Standard CPU runtime is sufficient; no GPU required.

* Installing dependencies may require a Runtime restart. You will be notified if a restart is required.
* Gate output in each run cell is the harness's live audit trail — FAIL entries in the naive runs are intentional; they demonstrate exactly what the gates catch.

### Library Installation

**Process:** Installs scikit-learn, xgboost, pandas, and numpy into the runtime. A flag file prevents reinstallation across the restart that follows.

**Outcome:** Runtime is ready for the harness. If the red banner appears below, restart the session and run all cells again.

In [ ]:
# ------------------------------
# LIBRARY INSTALLATION
# ------------------------------
# Installs scikit-learn, xgboost, pandas, numpy; flag file prevents repeat install post-restart.

import sys
from pathlib import Path
from IPython.display import display, HTML

flag_file = Path("/content/.deps_installed_flag") if Path("/content").exists() else Path(".deps_installed_flag")

if not flag_file.exists():
    print("Installing dependencies... this may take a minute or two.")
    import subprocess
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "scikit-learn", "xgboost", "pandas", "numpy"],
        check=True,
    )
    flag_file.touch()
    display(HTML('''
    <div style="border:4px solid #d93025;background:#fce8e6;color:#202124;padding:24px;border-radius:12px;font-family:Arial,sans-serif;margin:16px 0">
      <div style="font-size:30px;font-weight:800;color:#b31412;margin-bottom:12px">&#9888;&#65039; Runtime Restart Required</div>
      <div style="font-size:22px;line-height:1.5;font-weight:600">
        Dependencies have been successfully installed!<br><br>
        Please go to <b>Runtime &rarr; Restart session and run all</b> in the top menu.
      </div>
    </div>
    '''))
else:
    print("\u2705 Dependencies already installed. No restart required.")

### Harness and Data Loader Setup

**Process:** Downloads `gated_pipeline.py` from `EvagAIML/gated-pipeline` and the shared data loader from this case-study repo. Both are single-file fetches via GitHub raw URL — the harness has no pip package. If either file already exists locally (e.g., when running outside Colab with a pre-staged copy), the download is skipped.

**Outcome:** `gated_pipeline.py` and `_shared/data_access.py` are in the working directory, ready for import.

In [ ]:
# ------------------------------
# HARNESS AND DATA LOADER SETUP
# ------------------------------
# Downloads gated_pipeline.py and _shared/data_access.py from GitHub if not already present.
# The "if not exists" guard lets local runs (e.g., papermill) use a pre-staged copy.

import urllib.request
import sys
from pathlib import Path

gp_path = Path("gated_pipeline.py")
if not gp_path.exists():
    print("Downloading gated_pipeline.py from EvagAIML/gated-pipeline...")
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/EvagAIML/gated-pipeline/main/gated_pipeline.py",
        gp_path,
    )
else:
    print("gated_pipeline.py already present locally.")

shared_dir = Path("_shared")
da_path = shared_dir / "data_access.py"
if not da_path.exists():
    print("Downloading _shared/data_access.py from EvagAIML/000-gated-pipeline-cases...")
    shared_dir.mkdir(exist_ok=True)
    (shared_dir / "__init__.py").touch()
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/EvagAIML/000-gated-pipeline-cases/main/notebooks/_shared/data_access.py",
        da_path,
    )
else:
    print("_shared/data_access.py already present locally.")
    if not (shared_dir / "__init__.py").exists():
        (shared_dir / "__init__.py").touch()

if "." not in sys.path:
    sys.path.insert(0, ".")

print("\u2705 Harness and data loader ready.")

### Dataset Loading

**Process:** Downloads, SHA-256-verifies, and extracts the case-study dataset. `ensure_dataset` handles the full fetch-verify-cache cycle; subsequent runs skip straight to the cached copy.

**Analysis:** `GATED_PIPELINE_DATA_DIR` is set before `gated_pipeline` is imported because the harness resolves file paths at module-load time from that environment variable. Setting it afterwards would require a forced module reload.

**Outcome:** Both CSVs confirmed present: 8,763 retail rows and 8,880 Hill Country rows.

In [ ]:
# ------------------------------
# DATASET LOADING
# ------------------------------
# Downloads and verifies the dataset zip, sets the harness data path, imports the harness.
# Order matters: GATED_PIPELINE_DATA_DIR must be set before gated_pipeline is imported.

import os
import pandas as pd
from _shared.data_access import ensure_dataset

# Fetch and verify (idempotent — cached on re-runs)
data_dir = ensure_dataset("leak-detection__regression__gated-harness")

# Point the harness at the extracted CSVs before importing
os.environ["GATED_PIPELINE_DATA_DIR"] = str(data_dir / "raw")

import gated_pipeline as gp

# Confirm both files loaded correctly
retail_df = pd.read_csv(data_dir / "raw" / "retail_prediction.csv")
hill_df   = pd.read_csv(data_dir / "raw" / "hill_country_grocer_weekly_sales.csv")

print(f"retail_prediction.csv            — {len(retail_df):,} rows x {len(retail_df.columns)} columns")
print(f"hill_country_grocer_weekly_sales — {len(hill_df):,} rows x {len(hill_df.columns)} columns")
print(f"Retail target range:       ${retail_df['Product_Store_Sales_Total'].min():.0f}–${retail_df['Product_Store_Sales_Total'].max():.0f}")
print(f"Hill Country target range: ${hill_df['Weekly_Revenue_USD'].min():.2f}–${hill_df['Weekly_Revenue_USD'].max():.2f}")

### Retail — Naive Mode

**Process:** Runs the retail dataset with gates in audit-only mode. Before the train/test split, two target-derived features are injected: `te_Product_Type` (target-encoded by product type, fit on full data) and `avg_sales_for_product` (mean target by product ID, fit on full data). Gates log failures but do not block execution.

**Analysis:** Five gate failures appear in the output: `leakage.identifiers` (Product_Id included), `leakage.provenance` (target-derived features present), `leakage.degenerate` (`avg_sales_for_product` correlates 1.0 with target), `holdout.transform_scope` (encoder fit on all data), and `deploy.smoke_test` (model fails when the leaked columns are absent from a simulated inference request). The R²=1.0000 result is the tell — a perfect score on a real-world tabular dataset is not a success.

**Outcome:** Retail naive completes with R²=1.0000, MAPE=0.00%. The deploy gate confirms the model would fail immediately in production.

In [ ]:
# ------------------------------
# RETAIL — NAIVE MODE
# ------------------------------
# Gates are audit-only. Leaked features pass through to training unchecked.
# The R²=1.000 result comes from avg_sales_for_product (corr=1.0 with target).

gp.RUN_LOG.clear()
retail_naive = gp.run("retail", "naive")

print(f"\n  Naive result:  R\u00b2={retail_naive.get('r2', float('nan')):.4f}  "
      f"MAPE={retail_naive.get('mape', float('nan')):.2f}%  "
      f"status={retail_naive['status']}")

### Retail — Gated Mode

**Process:** Runs the retail dataset with gates enforced. The harness skips the target-derived feature injection step entirely in gated mode, fits the encoder on training data only, and verifies holdout integrity at each checkpoint.

**Analysis:** All gates pass. RandomForest (max_depth=9) wins the leaderboard and hypertuning converges on the first attempt at an interior grid point — no backtrack needed. The R²=0.9294 is the honest performance ceiling on the available features.

**Outcome:** Retail gated completes with R²=0.9294, MAPE=4.31% — 70.6 R² points below the naive score. That gap is entirely attributable to removing features that were fit with knowledge of the test set's answers.

In [ ]:
# ------------------------------
# RETAIL — GATED MODE
# ------------------------------
# Gates enforced. No target-derived feature injection; encoder fit on train split only.
# RandomForest depth=9 wins; hypertuning converges in one attempt.

gp.RUN_LOG.clear()
retail_gated = gp.run("retail", "gated")

print(f"\n  Gated result:  R\u00b2={retail_gated.get('r2', float('nan')):.4f}  "
      f"MAPE={retail_gated.get('mape', float('nan')):.2f}%  "
      f"status={retail_gated['status']}")

### Hill Country — Naive Mode

**Process:** Runs the Hill Country weekly sales dataset with gates audit-only. No features are planted — this run demonstrates in-data co-target leakage that exists naturally in the dataset: `Weekly_Units_Sold × price ≈ Weekly_Revenue_USD`.

**Analysis:** Three gate failures: `leakage.identifiers` (UPC included), `holdout.transform_scope` (encoder fit on full data), and `cotarget.reconstruction` — the scan finds `Promo_Price_USD × Weekly_Units_Sold` with correlation 0.9886 and `List_Price_USD × Weekly_Units_Sold` with 0.9766. Since no co-targets are declared in naive mode, both pairs remain in training. R²=0.9628 looks like a strong model; the co-target is why.

**Outcome:** Hill Country naive completes with R²=0.9628, MAPE=8.38%. The score would clear a production gate without question — that's the risk.

In [ ]:
# ------------------------------
# HILL COUNTRY — NAIVE MODE
# ------------------------------
# No planted leak; co-target leakage is structural (units x price = revenue).
# cotarget.reconstruction gate fires but is audit-only — pipeline runs to completion.

gp.RUN_LOG.clear()
hill_naive = gp.run("hill_country", "naive")

print(f"\n  Naive result:  R\u00b2={hill_naive.get('r2', float('nan')):.4f}  "
      f"MAPE={hill_naive.get('mape', float('nan')):.2f}%  "
      f"status={hill_naive['status']}")

### Hill Country — Gated Mode

**Process:** Runs the Hill Country dataset with gates enforced. Before any gate runs, the harness drops `UPC` (identifier) and `Weekly_Units_Sold` (declared co-target). The reconstruction scan then checks remaining features and finds no undeclared pairs that reconstruct revenue to the 0.95 threshold.

**Analysis:** All gates pass. Without `Weekly_Units_Sold`, the model must learn from prices, store attributes, product specs, and shelf placement — genuine demand signals rather than accounting arithmetic. HistGBR (max_depth=12) wins after one backtrack, where CV improvement below the 0.005 plateau threshold stopped the search.

**Outcome:** Hill Country gated completes with R²=0.9078, MAPE=14.55% — 5.5 points below the naive co-target score. The Tier 2 rating reflects that demand forecasting without the units column is genuinely harder; the honest performance ceiling is lower.

In [ ]:
# ------------------------------
# HILL COUNTRY — GATED MODE
# ------------------------------
# Weekly_Units_Sold declared as co-target, dropped before reconstruction scan.
# HistGBR depth=12 selected after one backtrack; all gates pass.

gp.RUN_LOG.clear()
hill_gated = gp.run("hill_country", "gated")

print(f"\n  Gated result:  R\u00b2={hill_gated.get('r2', float('nan')):.4f}  "
      f"MAPE={hill_gated.get('mape', float('nan')):.2f}%  "
      f"status={hill_gated['status']}")

### Results Comparison

**Process:** Collects all four run results into a summary table showing final test metrics, model, and Tier classification.

**Analysis:** The naive runs returned scores a production team would accept — 1.0000 and 0.9628. Neither generated an exception or an obviously bad metric. Only the gate log distinguishes them from deployable results. The gated scores are the actual performance ceilings.

**Outcome:** Retail leakage inflated R² by 0.0706 (7.1 points); Hill Country co-target inflated by 0.0550 (5.5 points). Both gated models are gate-certified and ready for further evaluation.

In [ ]:
# ------------------------------
# RESULTS COMPARISON
# ------------------------------
# Side-by-side summary of all four runs with R², MAPE, model, and Tier.

import pandas as pd

runs = [
    (retail_naive, "Retail — Naive"),
    (retail_gated, "Retail — Gated"),
    (hill_naive,   "Hill Country — Naive"),
    (hill_gated,   "Hill Country — Gated"),
]

rows = []
for r, label in runs:
    done = r["status"] == "completed"
    rows.append({
        "Run":   label,
        "R\u00b2":  f"{r.get('r2', float('nan')):.4f}" if done else "HALTED",
        "MAPE": f"{r.get('mape', float('nan')):.2f}%" if done else "—",
        "Model": r.get("model", "—"),
        "Tier":  f"T{r.get('tier','?')} ({r.get('label','')})" if done else "—",
    })

summary = pd.DataFrame(rows)
print(summary.to_string(index=False))

r_inf = retail_naive.get("r2", 0) - retail_gated.get("r2", 0)
h_inf = hill_naive.get("r2", 0)   - hill_gated.get("r2", 0)
print(f"\n  Retail inflation:       naive {retail_naive.get('r2',0):.4f} → gated {retail_gated.get('r2',0):.4f}  (\u0394 = {r_inf:.4f})")
print(f"  Hill Country inflation: naive {hill_naive.get('r2',0):.4f}   → gated {hill_gated.get('r2',0):.4f}  (\u0394 = {h_inf:.4f})")

## **Expanded Executive Summary**

### TLDR

The gated pipeline is the correct tool for any regression task where target-adjacent columns might be present in training data. On the retail dataset, naive evaluation returned R²=1.0000 — a result that should have triggered immediate investigation but passed every standard metric check. The gated run at R²=0.9294 is the actual performance ceiling. On the Hill Country dataset, naive R²=0.9628 carried a 0.989-correlated co-target (`Weekly_Units_Sold`); the gated R²=0.9078 is what demand forecasting on genuine signals produces.

The key finding is not the size of the gaps but what generates them. Neither naive score threw an error, and both looked like good models. The leakage was only visible in the gate log.

**Objective:** Demonstrate that the gated pipeline catches data leakage patterns that pass undetected through standard model evaluation. Two leakage types are covered: planted target-derived features (caught by `leakage.provenance` and `leakage.degenerate`) and structural co-target leakage where a training column algebraically reconstructs the target (caught by `cotarget.reconstruction`).

**Iterative Development:** The harness evaluates six candidate models per run and selects the leaderboard winner for hypertuning. On clean retail data, RandomForest (depth=9) converged on the first grid attempt. On Hill Country without the co-target, HistGBR (depth=12) required one backtrack — the first grid search edge-pinned at depth=8 and CV was still improving, so the grid expanded to [10, 12, 14] before converging at an interior point.

**Performance Analysis:** Retail naive (Ridge, R²=1.0000) scored perfectly on the test set through `avg_sales_for_product`, a feature encoding the mean target per product ID fit on the full dataset before the split — giving the model access to the answer before seeing the question. The `deploy.smoke_test` gate confirmed the model would fail in production: those features don't exist at inference time. Retail gated (RandomForest, R²=0.9294, MAPE=4.31%) is the real upper bound. Hill Country naive (RandomForest, R²=0.9628) looked Tier 2 — credible — but the model had essentially learned that revenue equals price times units, which is algebraically true and not a forecast. Hill Country gated (HistGBR, R²=0.9078, MAPE=14.55%) is the honest ceiling for weekly demand forecasting when inputs are prices, product specs, and store attributes.

**Economic Impact:** A retail model that scored R²=1.000 in evaluation and crashed on first production inference is an infrastructure reliability incident, not a model quality problem — rollback, root-cause investigation, data pipeline audit, retrain, and re-validation typically costs 2-4 days of senior engineering time per incident. A Hill Country demand forecast inflated by 5.5 R² points distorts safety-stock calculations, promotional planning, and reorder decisions before anyone identifies the source. The errors compound downstream.

**Deployment Readiness:** The gated retail model (R²=0.9294, MAPE=4.31%) is production-ready — it passes the smoke test on raw inference rows, the holdout is certified intact, and all features exist at inference time. The gated Hill Country model (R²=0.9078, MAPE=14.55%) requires volume-tier context before production: 14.55% MAPE on a high-velocity SKU and a slow-moving SKU represent different levels of operational error, and the aggregate number may mask poor performance on a specific segment.

**Next Steps:** Add volume-tier segmentation to the Hill Country evaluation (slow, medium, fast movers evaluated separately) to understand where the 14.55% MAPE concentrates. Backfill gate-log artifacts for existing registered models to establish a leakage baseline across the current model inventory. Integrate the harness as a pre-registration gate in the model registry workflow so future models arrive with gate logs attached.